## 2D synthetic experiment (Figure 1b)

`Run All` writes the two panels of Figure 1(b) into this folder:

- `loss_dynamics.pdf`: left panel, per-group training-loss dynamics over 30 epochs.
- `decision_boundary.pdf`: right panel, training points with the true, spurious, and learned boundaries (the learned boundary is drawn at epoch 1 and epoch 30).

The published right panel also has a rotation arrow and the "Epoch 1 / Epoch 30" text added by hand in an image editor; this notebook writes the un-annotated PDF.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

GROUP_COLORS = {
    "conf_nonspur": "#3f8fd5",
    "conf_spur": "#ff9933",
    "nonconf_nonspur": "#009e73",
    "nonconf_spur": "#b07adf",
    "other": "#d9dedc",
}
DOT_SIZE = 70
FONT_SIZE = 19


def make_synthetic_dataset_2d(n, p_spurious, sigma_small=0.0, seed=0,
                              k_confusing=250, k_nonconfusing=250):
    rng = np.random.RandomState(seed)
    z = rng.normal(size=n)
    x_core = np.tanh(0.7 * z).astype(np.float32)            # causal feature; label = sign(z)
    y = np.where(z >= 0.0, 1.0, -1.0)
    y01 = (y < 0).astype(np.float32)

    is_spurious = rng.rand(n) < p_spurious                  # spurious feature agrees with y w.p. p
    a = np.where(is_spurious, y, -y).astype(np.float32)
    m = rng.normal(1.0, 1.0 + sigma_small, size=n)
    while np.any(m <= 0):
        m[m <= 0] = rng.normal(1.0, 1.0 + sigma_small, size=int((m <= 0).sum()))
    X = np.stack([(a * m).astype(np.float32), x_core], axis=1)

    # near-boundary = small margin to the nearest opposite-class point
    X_pos, X_neg = X[y > 0], X[y < 0]
    margin = np.empty(n, dtype=np.float32)
    for i in range(n):
        diff = X[i] - (X_neg if y[i] > 0 else X_pos)
        margin[i] = np.sqrt(np.sum(diff * diff, axis=1)).min()
    order = np.argsort(margin)
    is_conf = np.zeros(n, dtype=bool)
    is_nonconf = np.zeros(n, dtype=bool)
    is_conf[order[:k_confusing]] = True
    is_nonconf[np.setdiff1d(order[-k_nonconfusing:], order[:k_confusing])] = True

    groups = {
        "conf_spur": np.where(is_conf & is_spurious)[0],
        "conf_nonspur": np.where(is_conf & ~is_spurious)[0],
        "nonconf_spur": np.where(is_nonconf & is_spurious)[0],
        "nonconf_nonspur": np.where(is_nonconf & ~is_spurious)[0],
    }
    return X, y01, groups


class LogisticReg(nn.Module):
    def __init__(self, input_dim=2):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
        nn.init.zeros_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        return self.linear(x).squeeze(-1)


def run_experiment(n, p_spurious, sigma_small=0.0, batch_size=256,
                   num_epochs=10, lr=1e-1, seed=0, capture_boundary=False):
    np.random.seed(seed)
    torch.manual_seed(seed)
    X, y01, groups = make_synthetic_dataset_2d(n, p_spurious, sigma_small, seed)

    X_t, y_t = torch.from_numpy(X), torch.from_numpy(y01)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)
    model = LogisticReg(X_t.size(1))
    criterion = nn.BCEWithLogitsLoss(reduction="none")
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    masks = {g: torch.zeros(n, dtype=torch.bool) for g in groups}
    for g, idx in groups.items():
        masks[g][idx] = True
    masks["other"] = ~torch.stack(list(masks.values())).any(0)

    loss_history = {g: [] for g in masks}

    def record():
        with torch.no_grad():
            losses = criterion(model(X_t), y_t)
        for g, mk in masks.items():
            loss_history[g].append(losses[mk].mean().item() if mk.any() else float("nan"))

    model.eval()
    record()
    weights = {}
    for epoch in range(1, num_epochs + 1):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            criterion(model(xb), yb).mean().backward()
            optimizer.step()
        model.eval()
        record()
        if capture_boundary:
            w = model.linear.weight[0].detach().numpy()
            weights[epoch] = (float(w[0]), float(w[1]), float(model.linear.bias.item()))

    if not capture_boundary:
        return loss_history, None
    boundary = {
        "weights": weights,
        "x_spur": X[:, 0],
        "x_causal": X[:, 1],
        "y_sign": np.where(y01 > 0.5, 1, -1),
        "groups": groups,
        "other_idx": np.where(masks["other"].numpy())[0],
    }
    return loss_history, boundary


In [ ]:
import math
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

GROUP_ORDER = ["conf_nonspur", "conf_spur", "nonconf_nonspur", "nonconf_spur", "other"]
GROUP_LABELS = ["Near-boundary & Non-spurious", "Near-boundary & Spurious",
                "Non-boundary & Non-spurious", "Non-boundary & Spurious", "Other Examples"]

# ---- left panel: per-group training-loss dynamics over 30 epochs ----
hist, _ = run_experiment(n=10000, p_spurious=0.9, sigma_small=0.0,
                         batch_size=128, num_epochs=30, lr=0.5, seed=0)
epochs = np.arange(len(hist["other"]))
plt.figure(figsize=(5, 5))
for key in GROUP_ORDER:
    plt.plot(epochs, hist[key], linewidth=4, color=GROUP_COLORS[key], linestyle="-")
plt.xlabel("Training epochs", fontsize=FONT_SIZE)
plt.ylabel("Average training loss", fontsize=FONT_SIZE)
plt.gca().set_box_aspect(1.0)
plt.tight_layout()
ax = plt.gca()
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.xticks(np.arange(0, len(epochs), 5), fontsize=FONT_SIZE)
ax.tick_params(axis="both", which="major", labelsize=FONT_SIZE)
ax.margins(0.05)  # default float margins keep x=0 and y=0 off the bottom-left corner
ax.legend([Patch(facecolor=GROUP_COLORS[k], edgecolor="none") for k in GROUP_ORDER], GROUP_LABELS,
          loc="lower right", bbox_to_anchor=(1.0, 1.02), ncol=1, frameon=True, fontsize=12,
          handlelength=1.5, handletextpad=0.6, labelspacing=0.3, borderpad=0.3, borderaxespad=0.0)
plt.savefig("loss_dynamics.pdf", bbox_inches="tight")
plt.close()
print("wrote loss_dynamics.pdf")

# ---- right panel: training points with true / spurious / learned boundaries ----
_, bnd = run_experiment(n=1000, p_spurious=0.9, sigma_small=0.0,
                        batch_size=128, num_epochs=30, lr=0.5, seed=0, capture_boundary=True)
x_spur, x_causal, y_sign = bnd["x_spur"], bnd["x_causal"], bnd["y_sign"]
groups, other_idx, weights = bnd["groups"], bnd["other_idx"], bnd["weights"]
spur_min, spur_max = float(x_spur.min()), float(x_spur.max())
causal_min, causal_max = float(x_causal.min()), float(x_causal.max())

# explicit pixel geometry so the saved panel matches the published figure exactly
W_R, H_R = 576, 670
PLOT_W, PLOT_H = 388, 446
LEFT, BOTTOM = 132, 84
FONT, PX = 21, 100.0

def scatter_group(ax, color, idxs):
    idxs = np.asarray(idxs)
    pos = idxs[y_sign[idxs] == 1]
    neg = idxs[y_sign[idxs] == -1]
    if len(pos):
        ax.scatter(x_spur[pos], x_causal[pos], s=DOT_SIZE, c=color, marker="+", linewidths=1.6)
    if len(neg):
        ax.scatter(x_spur[neg], x_causal[neg], s=DOT_SIZE, c=color, marker="_", linewidths=2.4)

fig = plt.figure(figsize=(W_R / PX, H_R / PX))
ax = fig.add_axes([LEFT / W_R, BOTTOM / H_R, PLOT_W / W_R, PLOT_H / H_R])
scatter_group(ax, GROUP_COLORS["other"], other_idx)
scatter_group(ax, GROUP_COLORS["nonconf_spur"], groups["nonconf_spur"])
scatter_group(ax, GROUP_COLORS["conf_nonspur"], groups["conf_nonspur"])
scatter_group(ax, GROUP_COLORS["conf_spur"], groups["conf_spur"])
scatter_group(ax, GROUP_COLORS["nonconf_nonspur"], groups["nonconf_nonspur"])
ax.axhline(y=0.0, color="black", linestyle="-", linewidth=1.5)
ax.axvline(x=0.0, color="green", linestyle="--", linewidth=1.5)
spur_vals = np.linspace(spur_min, spur_max, 200)
for ep in (1, 30):
    w_spur, w_causal, b = weights[ep]
    if abs(w_causal) > 1e-6:
        ax.plot(spur_vals, -(w_spur * spur_vals + b) / w_causal, color="red", linestyle="--", linewidth=2)
    elif abs(w_spur) > 1e-6:
        ax.axvline(x=-b / w_spur, color="red", linestyle="--", linewidth=2)
ax.set_xlim(spur_min, spur_max)
ax.set_ylim(causal_min, causal_max)
ax.set_xlabel("Spurious feature", fontsize=FONT)
ax.set_ylabel("Causal feature", fontsize=FONT)
x_ticks = np.arange(math.floor(spur_min), math.ceil(spur_max) + 1, 1)
ax.set_xticks(x_ticks[(x_ticks > spur_min) & (x_ticks < spur_max)])
y_ticks = np.arange(math.floor(causal_min * 4) / 4, math.ceil(causal_max * 4) / 4 + 0.25, 0.25)
ax.set_yticks(y_ticks[(y_ticks > causal_min) & (y_ticks < causal_max)])
ax.tick_params(axis="both", which="major", labelsize=FONT)
ax.legend([Line2D([0], [0], color="black", linestyle="-", linewidth=2),
           Line2D([0], [0], color="green", linestyle="--", linewidth=2),
           Line2D([0], [0], color="red", linestyle="--", linewidth=2)],
          ["True Boundary", "Spurious Boundary", "Learned Boundary"],
          loc="lower right", bbox_to_anchor=(1.0, 1.02), ncol=1, frameon=True, fontsize=12,
          handlelength=2.0, handletextpad=0.6, labelspacing=0.3, borderpad=0.3, borderaxespad=0.0)
fig.savefig("decision_boundary.pdf")
plt.close()
print("wrote decision_boundary.pdf")
